# **LangChain Essential Components Notebook**
This notebook contains essential LangChain components with simple working code examples.


## **1. Installation**

In [ ]:
!pip install langchain langchain-openai langchain-community faiss-cpu python-dotenv

## **2. Load API Key (.env)**

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
print('API Key Loaded:', OPENAI_API_KEY is not None)

## **3. Basic LLM Call**

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini')

response = llm.invoke('Explain machine learning in simple terms')
print(response.content)

## **4. Prompt Templates**

In [ ]:
from langchain.prompts import PromptTemplate

template = 'Explain {topic} in simple words'

prompt = PromptTemplate(
    input_variables=['topic'],
    template=template
)

prompt.format(topic='Neural Networks')

## **5. LCEL Chains**

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

chain.invoke({'topic': 'Deep Learning'})

## **6. Document Loader**

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('sample.txt')
documents = loader.load()

documents[:1]

## **7. Text Splitter**

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)
len(chunks)

## **8. Embeddings**

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

vector = embeddings.embed_query('What is AI?')
len(vector)

## **9. Vector Store (FAISS)**

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)

vectorstore

## **10. Retriever**

In [ ]:
retriever = vectorstore.as_retriever()

docs = retriever.invoke('Explain artificial intelligence')

docs[:1]

## **11. RAG (Retrieval Augmented Generation)**

In [ ]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)

qa_chain.invoke('Explain the document')

## **12. Memory (Conversation Memory)**

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

memory = ConversationBufferMemory()

conversation = ConversationChain(
    llm=llm,
    memory=memory
)

conversation.predict(input='Hello')

## **13. Tools**

In [ ]:
from langchain.tools import tool

@tool
def multiply(a:int, b:int) -> int:
    'Multiply two numbers'
    return a*b

multiply.invoke({'a':5,'b':6})

## **14. Agents**

In [ ]:
from langchain.agents import initialize_agent
from langchain.agents import AgentType

agent = initialize_agent(
    tools=[multiply],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

agent.run('What is 5 times 6?')